# Diagnostik Cache & Checkpoint

Cek cepat (tanpa GPU, tanpa load model) shape tiap file cache `.npy` dan tanggal terakhir diubah tiap
checkpoint `.pth` -- supaya kelihatan jelas mana yang masih berdasarkan data lama (24.954, bersih) dan
mana yang sudah ketimpa data baru (26.488, kotor / mengandung anomali).

In [1]:
import os
from datetime import datetime

import numpy as np


def describe_npy(path):
    if not os.path.exists(path):
        return f"  [TIDAK ADA] {path}"
    mtime = datetime.fromtimestamp(os.path.getmtime(path)).strftime("%Y-%m-%d %H:%M")
    try:
        arr = np.load(path, allow_pickle=True)
        if arr.dtype == object:
            shapes = [np.asarray(a).shape for a in arr]
            return f"  {path}\n    diubah: {mtime} | isi (object array): {shapes}"
        return f"  {path}\n    diubah: {mtime} | shape: {arr.shape}"
    except Exception as e:
        return f"  {path}\n    diubah: {mtime} | ERROR baca file: {e}"


def describe_ckpt(path):
    if not os.path.exists(path):
        return None
    mtime = datetime.fromtimestamp(os.path.getmtime(path)).strftime("%Y-%m-%d %H:%M")
    size_mb = os.path.getsize(path) / (1024 * 1024)
    return f"  {path}\n    diubah: {mtime} | ukuran: {size_mb:.1f} MB"

## Cache OOF/Test (non-TTA)

In [2]:
print("=== Cache dari notebook stacking (ConvNeXtV2-Tiny + SigLIP2) ===")
print(describe_npy("oof_probs_partial.npy"))
print(describe_npy("test_probs_partial.npy"))

print("\n=== Cache dari notebook model besar ===")
large_model_names = [
    "convnextv2_base.fcmae_ft_in22k_in1k",
    "convnextv2_large.fcmae_ft_in22k_in1k",
    "swinv2_large_window12to16_192to256.ms_in22k_ft_in1k",
]
for name in large_model_names:
    safe = name.replace("/", "_")
    print(describe_npy(f"oof_probs_large/{safe}_oof.npy"))
    print(describe_npy(f"test_probs_large/{safe}_test.npy"))

=== Cache dari notebook stacking (ConvNeXtV2-Tiny + SigLIP2) ===
  oof_probs_partial.npy
    diubah: 2026-07-21 02:24 | isi (object array): [(24954, 3), (24954, 3)]
  test_probs_partial.npy
    diubah: 2026-07-21 02:24 | isi (object array): [(1458, 3), (1458, 3)]

=== Cache dari notebook model besar ===
  oof_probs_large/convnextv2_base.fcmae_ft_in22k_in1k_oof.npy
    diubah: 2026-07-23 10:18 | shape: (26488, 3)
  test_probs_large/convnextv2_base.fcmae_ft_in22k_in1k_test.npy
    diubah: 2026-07-23 10:18 | shape: (1458, 3)
  oof_probs_large/convnextv2_large.fcmae_ft_in22k_in1k_oof.npy
    diubah: 2026-07-23 12:18 | shape: (26488, 3)
  test_probs_large/convnextv2_large.fcmae_ft_in22k_in1k_test.npy
    diubah: 2026-07-23 12:18 | shape: (1458, 3)
  oof_probs_large/swinv2_large_window12to16_192to256.ms_in22k_ft_in1k_oof.npy
    diubah: 2026-07-23 15:09 | shape: (26488, 3)
  test_probs_large/swinv2_large_window12to16_192to256.ms_in22k_ft_in1k_test.npy
    diubah: 2026-07-23 15:09 | shape: (1

## Cache TTA (kalau sudah pernah dijalankan)

In [3]:
print("=== Cache TTA (kalau ada) ===")
all_model_names = [
    "convnextv2_tiny.fcmae_ft_in1k",
    "vit_base_patch16_siglip_224.v2_webli",
] + large_model_names

for name in all_model_names:
    safe = name.replace("/", "_")
    print(describe_npy(f"oof_probs_tta/{safe}_oof_tta.npy"))
    print(describe_npy(f"test_probs_tta/{safe}_test_tta.npy"))

=== Cache TTA (kalau ada) ===
  [TIDAK ADA] oof_probs_tta/convnextv2_tiny.fcmae_ft_in1k_oof_tta.npy
  [TIDAK ADA] test_probs_tta/convnextv2_tiny.fcmae_ft_in1k_test_tta.npy
  [TIDAK ADA] oof_probs_tta/vit_base_patch16_siglip_224.v2_webli_oof_tta.npy
  [TIDAK ADA] test_probs_tta/vit_base_patch16_siglip_224.v2_webli_test_tta.npy
  [TIDAK ADA] oof_probs_tta/convnextv2_base.fcmae_ft_in22k_in1k_oof_tta.npy
  [TIDAK ADA] test_probs_tta/convnextv2_base.fcmae_ft_in22k_in1k_test_tta.npy
  [TIDAK ADA] oof_probs_tta/convnextv2_large.fcmae_ft_in22k_in1k_oof_tta.npy
  [TIDAK ADA] test_probs_tta/convnextv2_large.fcmae_ft_in22k_in1k_test_tta.npy
  [TIDAK ADA] oof_probs_tta/swinv2_large_window12to16_192to256.ms_in22k_ft_in1k_oof_tta.npy
  [TIDAK ADA] test_probs_tta/swinv2_large_window12to16_192to256.ms_in22k_ft_in1k_test_tta.npy


## Checkpoint (.pth) -- cek tanggalnya, bandingkan dgn kapan folder train sempat "kotor"

In [4]:
checkpoint_specs = [
    ("convnextv2_tiny.fcmae_ft_in1k", "."),
    ("vit_base_patch16_siglip_224.v2_webli", "."),
    ("convnextv2_base.fcmae_ft_in22k_in1k", "checkpoints_large"),
    ("convnextv2_large.fcmae_ft_in22k_in1k", "checkpoints_large"),
    ("swinv2_large_window12to16_192to256.ms_in22k_ft_in1k", "checkpoints_large"),
]

for name, ckpt_dir in checkpoint_specs:
    safe = name.replace("/", "_")
    print(f"\n{name}:")
    found_any = False
    for fold in range(5):
        p = os.path.join(ckpt_dir, f"{safe}_fold{fold}.pth")
        desc = describe_ckpt(p)
        if desc:
            found_any = True
            print(desc)
    if not found_any:
        print(f"  [TIDAK ADA checkpoint ditemukan di '{ckpt_dir}/']")


convnextv2_tiny.fcmae_ft_in1k:
  .\convnextv2_tiny.fcmae_ft_in1k_fold0.pth
    diubah: 2026-07-20 15:35 | ukuran: 106.4 MB
  .\convnextv2_tiny.fcmae_ft_in1k_fold1.pth
    diubah: 2026-07-20 16:27 | ukuran: 106.4 MB
  .\convnextv2_tiny.fcmae_ft_in1k_fold2.pth
    diubah: 2026-07-20 17:39 | ukuran: 106.4 MB
  .\convnextv2_tiny.fcmae_ft_in1k_fold3.pth
    diubah: 2026-07-20 18:43 | ukuran: 106.4 MB
  .\convnextv2_tiny.fcmae_ft_in1k_fold4.pth
    diubah: 2026-07-20 19:53 | ukuran: 106.4 MB

vit_base_patch16_siglip_224.v2_webli:
  .\vit_base_patch16_siglip_224.v2_webli_fold0.pth
    diubah: 2026-07-20 21:03 | ukuran: 354.4 MB
  .\vit_base_patch16_siglip_224.v2_webli_fold1.pth
    diubah: 2026-07-20 22:06 | ukuran: 354.4 MB
  .\vit_base_patch16_siglip_224.v2_webli_fold2.pth
    diubah: 2026-07-20 23:40 | ukuran: 354.4 MB
  .\vit_base_patch16_siglip_224.v2_webli_fold3.pth
    diubah: 2026-07-21 00:58 | ukuran: 354.4 MB
  .\vit_base_patch16_siglip_224.v2_webli_fold4.pth
    diubah: 2026-07-21

## Cara baca hasilnya

- **Shape cache OOF** harusnya `(24954, 3)` untuk semuanya. Kalau ada yang `(26488, 3)`, itu cache basi
  (dari saat folder masih kotor) -- perlu di-generate ulang.
- **Tanggal checkpoint**: bandingkan dengan kira-kira kapan folder `train/` sempat berisi 26.488 gambar.
  Kalau tanggal checkpoint SETELAH itu, kemungkinan bobot modelnya juga ikut kelatih dari data kotor --
  bukan cuma cache-nya yang salah, tapi model itu sendiri perlu dilatih ulang, bukan cuma re-inference.
- Kalau cuma **cache**-nya yang basi tapi **checkpoint**-nya ternyata dari sebelum folder kotor (tanggal
  lebih awal), cukup jalankan ulang `tta_inference_5model.ipynb` (atau bagian test/inference non-TTA) --
  itu akan re-generate cache dari checkpoint yang sudah benar + df yang sekarang sudah bersih (24.954),
  tanpa perlu training ulang.